# 2. 模式2: TypedDict
## 2.1 什么是 TypedDict

TypedDict 是 Python 3.8+ 引入的一种类型提示工具，即带有类型声明的字典结构。适合需要快速定义字典结构且无需 Pydantic 重量级功能的场景。

1. 普通 dict 没有类型信息：
```python
{
    "title": "盗梦空间",
    "year": 2010,
    "director": "克里斯托弗·诺兰",
    "rating": 9.3
}
```

2. TypeDict可以进一步说明:
* 这个字典应该有哪些字段
* 每个字段的类型是什么
* TypedDict 主要是类型声明，不是运行时强校验器。

In [1]:
from typing_extensions import TypedDict
class MovieDict(TypedDict):
    title: str
    year: int
    director: str
    rating: float
movie: MovieDict = {
    "title1": "盗梦空间",
    "year": 2010,
    "director": "克里斯托弗·诺兰",
    "rating": 8.8,
}
print(movie)

{'title1': '盗梦空间', 'year': 2010, 'director': '克里斯托弗·诺兰', 'rating': 8.8}


实例化字典时给出的字段名称和TypedDict不完全一致，此时IDE的静态类型检查会标记。

## 2.2 基本使用

Annotated的使用

> Annotated 用来在“类型”之外，再附加一些额外信息，即元数据。类似于 Pydantic 的 Field 。

基本形式：
```
Annotated[类型, 附加信息1, 附加信息2, ...]
```

In [2]:
import asyncio
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# 从.env文件中加载环境变量
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致

model = ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

1. 返回简单结构

In [10]:
"""
使用 TypedDict 模型定义结构化输出
"""
from typing_extensions import TypedDict, Annotated

class MovieTypedDict(TypedDict):
    """
    电影的详细信息
    """
    title: Annotated[str, "电影的正式名称，例如《盗梦空间》"]
    year: Annotated[int, "电影的公映年份，使用四位数字表示"]
    director: Annotated[str, "电影导演的全名"]
    rating: Annotated[float, "电影在10分制下的评分，可包含一位小数"]

# 设置模型结构化输出
structured_llm = model.with_structured_output(MovieTypedDict)

# 调用模型并获取结构化输出
response = structured_llm.invoke("给我介绍下电影《星际穿越》")

print(type(response))
print(response)

<class 'dict'>
{'title': '星际穿越', 'director': '克里斯托弗·诺兰', 'year': 2014, 'rating': 8.7}


与BaseModel不同，这里返回了字典而不是定义的类

注意：这个例子里，每个字段的 `Annotated` 只写了一个字符串。LangChain 会把 `Annotated` 的第 2 个参数当成**默认值**，所以这些文字实际上变成了默认值，而不是字段说明，详见 2.3.2。

举例2：返回嵌套结构

In [5]:
from typing import TypedDict, List, Annotated

# 使用TypedDict定义嵌套结构
class Actor(TypedDict):
    """演员情况"""
    name: Annotated[str, "演员姓名"]
    role: Annotated[str, "饰演的角色"]
class Movie(TypedDict):
    """电影情况"""
    title: Annotated[str, "电影标题"]
    year: Annotated[int, "上映年份"]
    director: Annotated[str, "导演"]
    cast: Annotated[List[Actor], "演员列表"] # 嵌套列表定义
    rating: Annotated[float, "评分"]
# 设置模型结构化输出
structured_llm = model.with_structured_output(Movie)
resp = structured_llm = structured_llm.invoke("给我介绍下电影《盗梦空间》")

# 访问嵌套数据
print(f"电影名: {resp['title']}")
print(f"上映年份: {resp['year']}")
print(f"导演: {resp['director']}")
print(f"演员列表:{resp['cast']}")
print(f"评分: {resp['rating']}")



电影名: 盗梦空间
上映年份: 2010
导演: 克里斯托弗·诺兰
演员列表:[{'name': '莱昂纳多·迪卡普里奥', 'role': '多姆·柯布'}, {'name': '约瑟夫·高登-莱维特', 'role': '亚瑟'}, {'name': '艾伦·佩吉', 'role': '阿里阿德涅'}, {'name': '汤姆·哈迪', 'role': '伊姆斯'}, {'name': '渡边谦', 'role': '齐藤'}, {'name': '玛丽昂·歌迪亚', 'role': '梅尔'}]
评分: 8.8


举例3：...的使用

说明：与模型提供商有关系

In [11]:
from typing_extensions import TypedDict, Annotated
class MovieDict(TypedDict):
    """
    电影的详细信息
    """
    title: Annotated[str, ..., "电影标题"]
    year: Annotated[int, ..., "电影上映年份"]
    director: Annotated[str, ..., "导演"]
    rating: Annotated[float, ..., "电影评分，满分十分"]

model_with_structure = model.with_structured_output(MovieDict)
response = model_with_structure.invoke("根据这段话抽取盗梦空间的信息，不包含的信息可以留空：盗梦空间在2010年上映，导演是克里斯托弗·诺兰。")
print(response)
print(type(response))

{'director': '克里斯托弗·诺兰', 'rating': 0, 'title': '盗梦空间', 'year': 2010}
<class 'dict'>


In [12]:
from langchain.chat_models import init_chat_model

model_deepseek = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
)
model_deepseek_with_structure = model_deepseek.with_structured_output(MovieDict)
response = model_deepseek_with_structure.invoke("根据这段话抽取盗梦空间的信息，不包含的信息可以留空：盗梦空间在2010年上映，导演是克里斯托弗·诺兰。")
print(response)
print(type(response))

{'title': '盗梦空间', 'year': 2010, 'director': '克里斯托弗·诺兰', 'rating': 0}
<class 'dict'>


在Python中，三个点 ... 被称为省略号（英文为 Ellipsis），它是一个内置的特殊对象（单例），其类型是 ellipsis，可以理解为占位符。下游框架（如LangChain）可以对 ... 作定制化处理，如LangChain中Annotated的 ... 表示当前字段是必须存在的，不可省略，用来指示模型的输出。

In [9]:
from typing_extensions import TypedDict, Annotated
class MovieDict(TypedDict):
    """
    电影的详细信息
    """
    title: Annotated[str, ..., "电影标题"]
    year: Annotated[int, ..., "电影上映年份"]
    director: Annotated[str, Ellipsis, "导演"]
    rating: Annotated[float, "电影评分，满分十分"]

model_with_structure = model.with_structured_output(MovieDict)
response = model_with_structure.invoke("根据这段话抽取盗梦空间的信息，不包含的信息必须留空：盗梦空间上映于2010年，我们并不知道它的导演是谁。")
print(response)
print(type(response))

{'title': '盗梦空间', 'year': 2010, 'director': '', 'rating': 0}
<class 'dict'>


director 和 rating 的信息都是缺失的，但是前者被标记为必填，因此输出的字典包含该字段但是值为空字符串。

## 2.3 为什么没有写 `...` 的 rating，模型也输出了？

上面的例子里，`rating` 没有写 `...`，模型却照样输出了 `'rating': 0`。先看 LangChain 实际发给模型的 schema（不请求模型）：

In [13]:
import json
from langchain_openai.chat_models.base import _convert_to_openai_response_format

# ChatOpenAI 默认的 json_schema 方式下，MovieDict 被转换成的 response_format（内部函数，仅用于观察）
response_format = _convert_to_openai_response_format(MovieDict)["json_schema"]

print("strict   =", response_format["strict"])
print("required =", response_format["schema"]["required"])
print()
for name, prop in response_format["schema"]["properties"].items():
    print(f"{name:<9}", json.dumps(prop, ensure_ascii=False))

strict   = False
required = ['title', 'year', 'director']

title     {"description": "电影标题", "type": "string"}
year      {"description": "电影上映年份", "type": "integer"}
director  {"description": "导演", "type": "string"}
rating    {"default": "电影评分，满分十分", "type": "number"}


### 2.3.1 从 schema 里读出的三件事

**1. `...` 只决定字段在不在 `required` 里。** `required` 的意思是"必须输出"。不在 `required` 里的意思是"可以不输出"，而**不是"不要输出"**。`rating` 仍然列在 `properties` 里，模型知道有这个字段，也可以输出它。GPT 这类模型倾向于把 schema 里列出的字段都填上，再加上提示词要求"不包含的信息必须留空"，它就给数字字段填了 0，当作"留空"。

**2. strict 是 `False`，没有人强制模型。** 02 笔记的 1.7 讲过：Pydantic 类走 `json_schema` 时，OpenAI SDK 会自动开启 strict，所有字段都变成必填。TypedDict 不经过 SDK 的这一步，而是由 LangChain 自己转成 schema，`strict` 默认为 `False`，所以 `required` 如实反映了 `...` 的写法。但不开 strict 时，`required` 对模型只是一个很强的提示，服务端并不强制；非必填字段输不输出，由模型自己决定。

**3. "电影评分，满分十分"变成了默认值。** 看 `rating` 那一行：它没有 `description`，却多了 `"default": "电影评分，满分十分"`，一个数字字段的默认值是一句中文。这是 `Annotated` 写法的一个陷阱，见 2.3.2。

### 2.3.2 Annotated 的参数顺序：类型、默认值、说明

LangChain 解析 TypedDict 时，把 `Annotated` 里类型后面的参数**按位置**理解：第 2 个是默认值，第 3 个是说明。源码（`langchain_core/utils/function_calling.py`）就是这一行：

```python
field_kwargs = dict(zip(("default", "description"), annotated_args[1:]))
```

所以只写一个字符串时，它会被当成**默认值**，而不是说明：

| 写法 | 默认值 | 字段说明 | 必填 |
|---|---|---|---|
| `title: str` | 无 | 无 | 是 |
| `Annotated[str, ...]` | 无 | 无 | 是 |
| `Annotated[str, ..., "电影标题"]` | 无 | 有 | 是 |
| `Annotated[float, "电影评分"]` | **"电影评分"**（被当成了默认值） | 无 | 否 |
| `Annotated[Optional[float], None, "电影评分"]` | `None` | 有 | 否 |

2.2 的第一个例子 `MovieTypedDict` 就踩了这个坑：每个字段都只写了一个字符串，结果所有文字都变成了默认值，schema 里连 `required` 都没有。模型照样输出了所有字段，原因和上面一样。

想要"非必填、有说明"，按表格最后一行写，把默认值 `None` 放在第 2 个位置：

In [14]:
from typing import Optional

class MovieDictFixed(TypedDict):
    """电影的详细信息"""
    title: Annotated[str, ..., "电影标题"]
    year: Annotated[int, ..., "电影上映年份"]
    director: Annotated[str, ..., "导演"]
    rating: Annotated[Optional[float], None, "电影评分，满分十分"]   # 非必填，有说明

for cls in (MovieTypedDict, MovieDictFixed):
    schema = _convert_to_openai_response_format(cls)["json_schema"]["schema"]
    print(f"{cls.__name__}: required = {schema.get('required')}")
    print("   rating ->", json.dumps(schema["properties"]["rating"], ensure_ascii=False))

MovieTypedDict: required = None
   rating -> {"default": "电影在10分制下的评分，可包含一位小数", "type": "number"}
MovieDictFixed: required = ['title', 'year', 'director']
   rating -> {"description": "电影评分，满分十分", "type": "number"}


### 2.3.3 那 `...` 到底有什么用？

`...` 管的是"能不能不输出"，区别要在模型真的省略字段时才看得出来。用修正后的 `MovieDictFixed`，并在提示词里明确要求"非必填字段不要输出"（会请求一次模型）：

In [15]:
structured_llm = model.with_structured_output(MovieDictFixed)
response = structured_llm.invoke(
    "根据这段话抽取盗梦空间的信息。原文没有提到的信息：必填字段留空，非必填字段直接不要输出。"
    "原文：盗梦空间上映于2010年，我们并不知道它的导演是谁。"
)
print(response)
print("包含 rating 吗:", "rating" in response)

{'director': '', 'rating': 0, 'title': '盗梦空间', 'year': 2010}
包含 rating 吗: True


同样的结构和提示词，换成 DeepSeek（会请求一次 DeepSeek）：

In [16]:
response_ds = model_deepseek.with_structured_output(MovieDictFixed).invoke(
    "根据这段话抽取盗梦空间的信息。原文没有提到的信息：必填字段留空，非必填字段直接不要输出。"
    "原文：盗梦空间上映于2010年，我们并不知道它的导演是谁。"
)
print(response_ds)
print("包含 rating 吗:", "rating" in response_ds)

{'title': '盗梦空间', 'year': 2010, 'director': ''}
包含 rating 吗: False


两个模型的做法不一样：

- **GPT** 仍然输出了 `rating: 0`，即使提示词明确要求"非必填字段不要输出"
- **DeepSeek** 省略了非必填的 `rating`，而必填的 `director` 仍然输出了（空字符串）

这就是 `...` 的真正作用：**必填字段一定要出现**（开了 strict 时由服务端保证；没开时是一个很强的提示，模型通常会遵守），**非必填字段可能出现，也可能不出现**，由模型自己决定。

所以读取非必填字段时，要写 `response.get("rating")`，不能直接写 `response["rating"]`，否则字段缺失时会抛出 `KeyError`。

## 2.4 TypedDict 与 Pydantic 对比

两种写法描述的是同一件事：有哪些字段、什么类型、什么含义、是否必填。区别主要在**拿到模型的回复之后**：Pydantic 会创建对象、做校验；TypedDict 只把 JSON 解析成字典，不做任何检查。

用同一段"有问题的" JSON 对比一下（不请求模型）：`year` 写成了中文，`rating` 缺失。

In [17]:
from pydantic import BaseModel, Field, ValidationError

bad_json = '{"title": "盗梦空间", "year": "二零一零", "director": ""}'

# TypedDict：with_structured_output 用的解析器是 JsonOutputParser，只负责把 JSON 转成字典
typed_dict_parser = model.with_structured_output(MovieDictFixed).last
print("TypedDict 的解析器:", type(typed_dict_parser).__name__)
print("TypedDict 的结果:  ", typed_dict_parser.parse(bad_json))
print()

# Pydantic：同样的字段，用 BaseModel 定义
class MovieModel(BaseModel):
    """电影的详细信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="电影上映年份")
    director: str = Field(description="导演")
    rating: Optional[float] = Field(default=None, description="电影评分，满分十分")

try:
    MovieModel.model_validate_json(bad_json)
except ValidationError as e:
    error = e.errors()[0]
    print("Pydantic 的结果:   ValidationError ->", error["loc"], error["msg"])

# year 改对之后：缺失的 rating 由 Pydantic 补上默认值
print("Pydantic 补默认值: ", repr(MovieModel.model_validate_json('{"title": "盗梦空间", "year": 2010, "director": ""}')))

TypedDict 的解析器: JsonOutputParser
TypedDict 的结果:   {'title': '盗梦空间', 'year': '二零一零', 'director': ''}

Pydantic 的结果:   ValidationError -> ('year',) Input should be a valid integer, unable to parse string as an integer
Pydantic 补默认值:  MovieModel(title='盗梦空间', year=2010, director='', rating=None)


TypedDict 把错误的 `year` 原样交了出来，缺失的 `rating` 也没有补；Pydantic 则拦下了错误的类型，并给缺失的字段补上了默认值。完整对比：

| 对比项 | Pydantic `BaseModel` | TypedDict |
|---|---|---|
| 返回值 | 模型类的实例，用 `result.title` 访问 | 普通字典，用 `result["title"]` 访问 |
| 字段说明 | `Field(description=...)` | `Annotated[类型, 默认值, 说明]`，注意参数位置 |
| 必填 | 没有默认值的字段就是必填 | 不写 `Annotated`，或者第 2 个参数写 `...` |
| 类型校验 | 有，类型不对抛出 `ValidationError` | 无，模型返回什么就是什么 |
| 类型转换 | 有，如 `"2010"` → `2010`、`"2026-10-01"` → `date` | 无 |
| 缺失的字段 | 用默认值补上 | 字典里就没有这个键 |
| 约束条件（`ge`、`min_length` 等） | 支持 | 不支持，LangChain 只读取 `Annotated` 里的默认值和说明 |
| 自定义校验器 | 支持 | 不支持 |
| ChatOpenAI 默认的 strict | 开启，所有字段都变成必填（02 笔记 1.7） | 不开启，由 `...` 决定哪些字段必填 |
| 适合的场景 | 需要可靠数据的正式项目 | 快速试验，或者结果本来就要当字典用，并且能接受不做校验 |

## 2.5 总结

1. TypedDict 用"带类型声明的字典"描述输出结构，`with_structured_output` 返回的是**普通字典**。
2. 字段说明写在 `Annotated` 里，参数顺序是**类型、默认值、说明**。只写一个字符串时，它会被当成默认值；想要"非必填、有说明"，写成 `Annotated[Optional[类型], None, "说明"]`。
3. `...` 只决定字段是否必填，也就是在不在 `required` 里。非必填不等于不输出：输不输出由模型决定，GPT 倾向于都填上，DeepSeek 可能会省略。
4. TypedDict 在 ChatOpenAI 下不开启 strict，结果也不经过任何校验：类型错了、字段缺了，都要自己检查。读取非必填字段时用 `result.get("字段名")`。
5. 需要校验、类型转换、默认值、约束条件时用 Pydantic；只想快速拿到一个字典时用 TypedDict。